In [ ]:
# Tåg och trafikanalys Stockholm

Importering av bibliotek

Här importerar jag de bibliotek som jag kommer ha användning för detta projekt vilket inkluderar os, csv, requests, matplotlib och datetime.

In [ ]:
# ==============================================================================================================
# Importering av standard- och externa bibliotek
# ==============================================================================================================

import csv      # Hjälper med att skapa och skriva till CSV-filer
import os       # Använder för att kontrollera saker på datorn t.ex. om en fil finns eller inte
from datetime import datetime       # Används för att ha koll på datum och tid samt används för att räkna ut tidsskillnader för tågen

import requests         # Används för att hämta data från webben
import matplotlib.pyplot as plt     # Används för att skapa mitt diagram

print("Importeringen av bibliotek klar!")

API-Nyckeln

In [ ]:
# ==============================================================================================================
# API-nyckel
# ==============================================================================================================

API_KEY = "ad1dd6ea9368474cb498c74084e8573e"        # Skapar en variabel som innehåller API-nyckeln som behövs för att hämta data från Trafiklab API:et
BASE_URL = "https://realtime-api.trafiklab.se/v1"   # Skapar en variabel som innehåller URL:en för Trafiklab API:et

Föräldrar klass

In [ ]:
# ==============================================================================================================
# Föräldraklass
# ==============================================================================================================

# Föräldraklass som representerar en grundläggande avgång
class Departure:

    def __init__(self, station, line, destination, time, delay_seconds):        # __init__ är konstruktorn och station, line, destination, time och delay_seconds är parametrar (som är variabler)
        self.station = station      # Sparar stationsnamnet på objektet
        self.line = str(line)       # Sparar linjenumret som en sträng på objektet
        self.destination = destination      # Sparar destinationsnamnet på objektet
        self.time = time        # Sparar avgångstiden på objektet
        self.delay_seconds = delay_seconds      # Sparar förseningstiden i sekunder på objektet

    # Definierar metoden som konverterar försening från sekunder till hela minuter.
    def get_delay_minutes(self):

        if self.delay_seconds is None:      # Kontrollerar om förseningsvärdet saknar (None) och då retunerar den 0
            return 0
        return round(self.delay_seconds / 60)       # Delar upp sekunderna med 60 för att få fram minuter och avrundar sedan värdet i heltal mha round



In [ ]:
# ==============================================================================================================
# Barnklass för tågavgångar och statuslogik
# ==============================================================================================================

# Barnklass som ärver alla egenskaper från Departure och lägger till statuslogik
class TrainDeparture(Departure):

    # Metod som returnerar statusen för avgången
    def get_status(self):

        # Returnerar statusen på tåget i text
        delay = self.get_delay_minutes()

        # Om tåget är mer än 0 minuter sen så returneras "Försenad" med antal minuter, annars returneras "I tid"
        if delay > 0:
            return f"Försenad ({delay} min)"
        return "I tid"

# ==============================================================================================================

# Barnklass specifikt för pendeltåg med aktiv valideringslogik och ärver från barnklassen TrainDeparture
class CommuterTrainDeparture(TrainDeparture):

    # Ett set med  pendeltågslinjer i SL-trafiken i Stockholm
    PENDELTAG_LINES = {"40", "41", "42", "43", "43X", "44", "48"}

    # Validerar om avgången tillhör pendeltågstrafiken baserat på linjenummer (40-serien) och transportslag.
    @classmethod        # Hjälpfunktion som används för att avgöra om en avgång är ett pendeltåg eller inte. gör också att metoden tillhör klassen och inte ett specifikt objekt av klassen
    
    def is_pendeltag(cls, line_designation, transport_mode="TRAIN", group_of_lines=""):     # Definierar en klassmetod som tar in linjedesignation, transportläge och grupp av linjer som argument. Den returnerar True om det är ett pendeltåg, annars False.
        line_str = str(line_designation).strip()            # Tar bort eventuella extra mellanslag från linjedesignationen och konverterar den till en sträng
        mode_str = str(transport_mode).strip().upper()      # Omvandlar transportslaget till stora bokstäver och tar bort eventuella extra mellanslag
        group_str = str(group_of_lines).strip().lower()     # Omvandlar gruppen av linjer till små bokstäver och tar bort eventuella extra mellanslag

        # Avvisa uttryckligen buss, t-bana, spårvagn och färja
        if mode_str in ["BUS", "METRO", "TRAM", "FERRY", "SHIP"]:      
            return False

        # Om group_of_lines i API:et anger pendeltåg
        if "pendeltåg" in group_str or "commuter" in group_str:
            return True

        # Validera mot SL:s pendeltågslinjer (40–48)
        if line_str in cls.PENDELTAG_LINES:
            return True

        # Kollar om linjen består av siffror och ligger mellan 40-48
        if line_str.isdigit() and 40 <= int(line_str) <= 48:
            return True

        # Retunerar False om inget av villkoren ovan uppfylldes
        return False

Funktioner

In [ ]:
# ==============================================================================================================
# Funktioner för att hämta data från API
# ==============================================================================================================
import requests

# Funktionen som hämtar avgångar direkt från SL:s öppna API utan nyckel.
def fetch_departures(site_id):
  
    url = f"https://transport.integration.sl.se/v1/sites/{site_id}/departures"

    # Hämtar data från API och har felhantering om något går fel
    try:
        response = requests.get(url, timeout=10)        # Kopplar upp sig mot API och om det tar mer än 10 sekunder avbryter den sin förfrågning
        response.raise_for_status()     # Genererar fel om anropet till API:n misslyckades
        return response.json()      # Retunerar datan i JSON-format (liknande python-dictionary och listor), ett standardformat för att skicka data mellan klient och server
   
    except Exception as e:      # Fångar upp eventuella fel under API anropet och sparar det 
        print(f"Fel vid hämtning från API: {e}") # Betyder att om något går fel vid hämtning av data från API:et så skrivs ett felmeddelande ut med information om felet
        return None     # Om ett fel inträffade så retunerar den None



In [ ]:
# ==============================================================================================================
# Funktion för att filtrera/sortera pendeltågsavgångar
# ==============================================================================================================

def filter_pendeltag_departures(api_data, station_name):

    pendeltag_list = []    # Skapar en tom lista
    
    if not api_data or "departures" not in api_data:     # Kollar om API data saknar eller "departures"
        return pendeltag_list       # Retunerar den tomma listan om ingen data hittades

    for item in api_data["departures"]:     # Loopar igenom alla avgångar i API svaret
        line_info = item.get("line", {})    # Hämtar info om pendellinjen
        transport_mode = line_info.get("transport_authority", "") or line_info.get("group_of_lines", "")    # Hämtar transportslag/grupp
        line_no = str(line_info.get("designation", ""))     # Hämtar linjenumret som i en string
        
        # Kontrollera om det är ett pendeltåg mha klassmetoden för att avgöra om det är ett pendeltåg
        if CommuterTrainDeparture.is_pendeltag(line_no, "TRAIN", transport_mode):
            destination = item.get("destination", "Okänd destination")      # Hämtar tågets destination
            time_str = item.get("scheduled", "Okänd tid")       # Hämtar schemalagd avgångstid
            
            # Beräkna försening i sekunder
            delay_seconds = 0      # Börjar med att förseningen är o sekunder
            if "expected" in item and "scheduled" in item:      # Kollar om förväntad och schemalagd tid finns i datan
                try:
                    sched = datetime.fromisoformat(item["scheduled"])       # Omvandlar schemalagd tid från sträng till ett datetime-objekt
                    exp = datetime.fromisoformat(item["expected"])      # Omvandlar förväntad tid från sträng till ett datetime-objekt
                    delay_seconds = max(0, int((exp - sched).total_seconds()))      # Beräknar skillnaden mellan schemalagd och förväntad tid, värdet får ej bli negativt
                except ValueError:     
                    delay_seconds = 0       # Om det skulle bli formateringsfel på tiden så sätts förseningen till 0 sekunder

            # Skapar ett CommuterTrainDeparture-objekt och lägger till alla värden i listan
            dep_obj = CommuterTrainDeparture(
                station=station_name,
                line=line_no,
                destination=destination,
                time=time_str,
                delay_seconds=delay_seconds
            )
            pendeltag_list.append(dep_obj)

    return pendeltag_list[:10]  # Returnerar max 10 avgångar



In [ ]:
# ==============================================================================================================
# Funktion för att spara data till CSV
# ==============================================================================================================

def save_to_csv(departures, filename="trafikdata.csv"):     # Öppnar och skriver till en CSV-fil med avgångsdata

    try:
        file_exists = os.path.exists(filename)      # Kontrollerar om filen redan finns och sparar resultatet i en variabel
        with open(filename, "w", newline="", encoding="utf-8") as file: # Ändrat från "a" till "w" för att skriva över filen varje gång
            writer = csv.writer(file)       # Skapar en CSV-skrivare som kan skriva till filen
            
            if not file_exists:
                # Skriv rubrikerna Station, Linje, Desrtination, Tid, Försening (min), Status i CSV-filen
                writer.writerow(["Station", "Linje", "Destination", "Tid", "Försening (min)", "Status"])

            # Loopar igenom avgångarna och skriver varje avgångsdata till CSV-filen
            for dep in departures:
                writer.writerow([
                    dep.station, 
                    dep.line, 
                    dep.destination, 
                    dep.time, 
                    dep.get_delay_minutes(), 
                    dep.get_status()
                ])
        print(f"\nData sparades till {filename}!")
        
    except IOError as e:     # Fångar upp eventuella fel vid filhantering och sparar det
        print(f"Filfel: Kunde inte spara datan. {e}")


In [ ]:
# ==============================================================================================================
# Funktion för att plotta förseningar
# ==============================================================================================================

def plot_delays(departures, station_name):

    display_departures = departures[:10]    # Tar max 10 avgångar för att visa i diagrammet

    labels = []     # Skapar en tom lista för att lagra etiketter för x-axeln
    delays = []     # Skapar en tom lista för att lagra förseningar i minuter

    # Loopar igenom avgångarna och skapar etiketter och förseningar för diagrammet
    for dep in display_departures:
        short_time = dep.time[11:16] if len(dep.time) >= 16 else dep.time   # Plockar ut tiden i formatet HH:MM från avgångstiden, om avgångstiden är kortare än 16 tecken används hela strängen
        labels.append(f"{short_time}\nLinje {dep.line}")        # Skapar etiketter med tid och linje för x-axeln
        delays.append(dep.get_delay_minutes())      # Skapar en lista med förseningar i minuter för y-axeln

    plt.figure(figsize=(10, 5))     # Skapar en figur med storleken 10x5 tum
    plt.bar(labels, delays)     # Skapar ett stapeldiagram med etiketter och förseningar

    plt.title(f"Förseningar vid {station_name}")
    plt.xlabel("Tid och linje")
    plt.ylabel("Försening i minuter")

    # Ifall om alal tåg är itid
    if max(delays, default=0) == 0:
        plt.ylim(0, 10)

    plt.tight_layout()      # Justerar layouten för att undvika överlappning av etiketter och titlar
    plt.show()      # OBS! Om alla tågeen är i tid är diagrammet tomt, men det är fortfarande bra att visa det för att bekräfta att allt är i tid

    # Om alla avgångar är i tid, skriv ut ett meddelande
    if all(delay == 0 for delay in delays):
        print(f"Alla avgångar vid {station_name} är i tid!")

Menyn

Här kan användaren välja vad progrmamet ska göra. Antingen analysera dagens avgångar (alternativ 1) eller analysera morgondagens avgångar (alternativ 2).

In [ ]:

# ==============================================================================================================
# Meny och användarinteraktion
# ==============================================================================================================

# Sidans ID:n för pendeltågsstationerna
STOCKHOLM_CITY_ID = "9509"   # Stockholm City
ODENPLAN_ID = "9117"         # Stockholm Odenplan

# ==============================================================================================================
# Meny funktionen
def main():
    while True:     # Använder while True-loop så menyn visas om och om igen tills användaren väljer att avsluta
        print("\n========================================")
        print("Pendeltågsanalys - Huvudmeny")
        print("========================================")
        print("1. Hämta pendeltåg för Stockholm City")
        print("2. Hämta pendeltåg för Stockholm Odenplan")
        print("3. Avsluta")
        
        choice = input("\nVälj ett alternativ (1-3): ").strip()
        
        if choice == "1":
            print("\nHämtar aktuell pendeltågsdata för Stockholm City...")

            # Anropar funktionen fetch_departures med STOCKHOLM_CITY_ID för att hämta avgångsdata
            data = fetch_departures(STOCKHOLM_CITY_ID)

            # Anropar funktionen filter_pendeltag_departures för att filtrera ut pendeltågsavgångar för Stockholm City
            departures = filter_pendeltag_departures(data, "Stockholm City")
            
            if departures:      # Om listan innehåller avgångar, skriv ut dem i terminalen
                print("-" * 50)

                # Loopar igenom avgångarna och skriver ut information om varje avgång
                for dep in departures:    
                    print(f"Klockan {dep.time[11:16]} | Linje {dep.line} mot {dep.destination} -> {dep.get_status()}")
                
                print("-" * 50)
                save_to_csv(departures)
                plot_delays(departures, "Stockholm City")
            
            else:
                print("Hittade inga aktiva pendeltågsavgångar för den valda stationen.")

        elif choice == "2":
            print("\nHämtar aktuell pendeltågsdata för Stockholm Odenplan...")

            data = fetch_departures(ODENPLAN_ID)        # Anropar funktionen fetch_departures med ODENPLAN_ID för att hämta avgångsdata
            departures = filter_pendeltag_departures(data, "Stockholm Odenplan")        # Anropar funktionen filter_pendeltag_departures för att filtrera ut pendeltågsavgångar för Stockholm Odenplan
            
            if departures:    # Om listan innehåller avgångar, skriv ut dem i terminalen
                print("-" * 50)

                for dep in departures:
                    print(f"Klockan {dep.time[11:16]} | Linje {dep.line} mot {dep.destination} -> {dep.get_status()}")
                print("-" * 50)
                save_to_csv(departures)
                
                plot_delays(departures, "Stockholm Odenplan")
            else:
                print("Hittade inga aktiva pendeltågsavgångar för den valda stationen.")

        elif choice == "3":
            print("Avslutar programmet.")
            break

# Anropar main-funktionen för att starta programmet
main() 